# 🧪 TelcoPulse: Advanced Statistical Experimentation & CUPED Variance Reduction

This notebook provides the mathematical rigor, synthetic experiment simulations, and empirical benchmarks for:
1. **CUPED (Controlled-experiment Using Pre-Experiment Data)** for variance reduction and 40%+ sample size efficiency.
2. **Difference-in-Differences (DiD)** 2-Way Fixed Effects Quasi-Experimentation for non-randomized policy rollouts.
3. **mixture Sequential Probability Ratio Testing (mSPRT)** for continuous experimentation without $p$-hacking.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

# Import custom TelcoPulse experimentation engines
import sys, os
sys.path.append(os.path.abspath('..'))
from src.ab_testing import CUPEDExperimentEngine, DifferenceInDifferencesEngine, MSPRTExperimentEngine

np.random.seed(42)
print("✅ Loaded TelcoPulse experimentation modules.")

## 1. Mathematical Derivation: CUPED Variance Reduction

Let $Y_i$ denote the primary business outcome (e.g. 30-day customer retention or revenue), and $X_i$ denote the pre-experiment baseline metric (e.g. historical ARPU or usage) measured prior to treatment assignment.

The CUPED estimator constructs an adjusted metric:
$$\hat{Y}_{i, \text{cuped}} = Y_i - \theta (X_i - E[X])$$

The variance of the treatment effect estimate $\hat{\Delta}_{\text{cuped}} = \bar{Y}_{T, \text{cuped}} - \bar{Y}_{C, \text{cuped}}$ is minimized when:
$$\theta^* = \frac{\text{Cov}(Y, X)}{\text{Var}(X)}$$

Plugging in $\theta^*$ yields:
$$\text{Var}(\hat{Y}_{\text{cuped}}) = \text{Var}(Y) \cdot \left(1 - \text{Corr}(Y, X)^2\right)$$

When $\text{Corr}(Y, X) = 0.65$, variance is reduced by $\approx 42.25\%$, equivalent to running the experiment with **$1.73\times$ larger sample size**!

In [ ]:
# 1. Simulate Pre-Experiment and Post-Experiment Customer Telemetry
n_customers = 5000

# Pre-experiment baseline ARPU ($)
x_pre = np.random.gamma(shape=5.0, scale=12.0, size=n_customers)

# Random 50/50 A/B split
treatment_mask = np.random.binomial(1, 0.5, size=n_customers).astype(bool)

# Treatment effect: +$4.50 ARPU lift with heteroskedastic noise
true_lift = 4.50
noise = np.random.normal(0, 14.0, size=n_customers)

# Post-experiment ARPU (strongly correlated with pre-experiment baseline)
y_post = 0.82 * x_pre + 10.0 + noise
y_post[treatment_mask] += true_lift

x_treat, y_treat = x_pre[treatment_mask], y_post[treatment_mask]
x_ctrl, y_ctrl = x_pre[~treatment_mask], y_post[~treatment_mask]

print(f"Treatment Group: N={len(y_treat)}, Control Group: N={len(y_ctrl)}")
print(f"Pre-Post Correlation: r = {np.corrcoef(x_pre, y_post)[0, 1]:.4f}")

In [ ]:
# 2. Run CUPED Evaluation
results = CUPEDExperimentEngine.evaluate_cuped(x_treat, y_treat, x_ctrl, y_ctrl)

print("=== RAW A/B TEST RESULTS ===")
print(f"Absolute Lift: ${results['raw_ab_test']['absolute_effect']:.2f}")
print(f"Std Error: {results['raw_ab_test']['standard_error']:.4f}")
print(f"P-value: {results['raw_ab_test']['p_value']:.5f}")
print(f"95% CI: {results['raw_ab_test']['ci_95']}")

print("\n=== CUPED VARIANCE-REDUCED A/B TEST RESULTS ===")
print(f"Absolute Lift: ${results['cuped_ab_test']['absolute_effect']:.2f}")
print(f"Std Error: {results['cuped_ab_test']['standard_error']:.4f}")
print(f"P-value: {results['cuped_ab_test']['p_value']:.5f}")
print(f"95% CI: {results['cuped_ab_test']['ci_95']}")
print(f"Variance Reduction: {results['variance_reduction_pct']}%")
print(f"Sample Size Savings: {results['sample_size_savings_pct']}%")
print(f"Effective Sample Multiplier: {results['effective_sample_multiplier']}x")

## 2. Difference-in-Differences (DiD) 2-Way Fixed Effects

For quasi-experimental rollouts (e.g. testing proactive retention offers in specific regions without individual randomization), we fit the 2-Way Fixed Effects model:
$$Y_{it} = \beta_0 + \beta_1 \text{Treated}_i + \beta_2 \text{Post}_t + \beta_3 (\text{Treated}_i \times \text{Post}_t) + \epsilon_{it}$$
where $\beta_3 = \text{ATT}$ (Average Treatment Effect on the Treated).

In [ ]:
# Simulate 2-Period Quasi-Experiment
n_region = 1200
ctrl_pre = np.random.normal(50.0, 8.0, n_region)
ctrl_post = ctrl_pre + np.random.normal(2.0, 4.0, n_region) # macro trend +$2

treat_pre = np.random.normal(52.0, 8.0, n_region)
# Treatment region experiences macro trend +$2 PLUS true policy effect +$6.50
treat_post = treat_pre + 2.0 + 6.50 + np.random.normal(0, 4.0, n_region)

did_res = DifferenceInDifferencesEngine.fit_did(ctrl_pre, ctrl_post, treat_pre, treat_post)

print("=== DIFFERENCE-IN-DIFFERENCES REGRESSION ===")
print(f"Estimated Policy ATT: ${did_res['causal_att']:.2f} (SE: {did_res['standard_error']:.3f})")
print(f"T-stat: {did_res['t_statistic']}, P-val: {did_res['p_value']:.5f}")
print(f"95% Confidence Interval: {did_res['ci_95']}")
print(f"Statistically Significant: {did_res['statistically_significant']}")

## 3. mixture Sequential Probability Ratio Testing (mSPRT)

Continuous real-time experimentation without Type-I error inflation ($p$-hacking protection).
Likelihood Ratio Martingale threshold: $\Lambda_n \ge 1 / \alpha = 20.0$ for $\alpha=0.05$.

In [ ]:
msprt = MSPRTExperimentEngine(alpha=0.05, mixing_variance_tau2=0.5)

# Simulate sequential stream of incoming subscriber pairs
stream_size = 1000
ctrl_stream = np.random.normal(0.0, 1.0, stream_size).tolist()
treat_stream = (np.random.normal(0.18, 1.0, stream_size)).tolist() # true positive delta

lambdas = []
stopped_step = None

for step in range(10, stream_size, 5):
    res = msprt.evaluate_stream(treat_stream[:step], ctrl_stream[:step])
    lambdas.append((step, res['current_lr_martingale']))
    if res['stopped_early'] and stopped_step is None:
        stopped_step = step
        print(f"⚡ mSPRT Early Stopping Triggered at sample N={step}!")
        print(f"Martingale Lambda: {res['current_lr_martingale']} >= {res['stopping_threshold']}")
        print(f"Decision: {res['decision']}")
        break

print("✅ mSPRT continuous experimentation benchmark complete.")